# U5-02 · Árboles, conjuntos y SVM

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 5, apartados 5.4 a 5.6*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
yc = 1 - yc                                   # 1 = maligno
Xc_ent, Xc_pru, yc_ent, yc_pru = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=0)
print(len(yc_ent), "tumores de entrenamiento,", yc_ent.sum(), "malignos")

## 1. La impureza de Gini a mano (ejemplo 5.4)

In [ ]:
def gini(*recuentos):
    n = sum(recuentos)
    return 1 - sum((k / n) ** 2 for k in recuentos)

raiz = gini(267, 159)
izq, der = gini(250, 15), gini(17, 144)
ponderada = (265 * izq + 161 * der) / 426
print(f"raíz {raiz:.3f} · izquierda {izq:.3f} · derecha {der:.3f} · ponderada {ponderada:.3f} · reducción {raiz - ponderada:.3f}")

## 2. Un árbol de profundidad 2 (figura 5.4)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
arbol = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xc_ent, yc_ent)
plt.figure(figsize=(10, 5))
plot_tree(arbol, feature_names=list(Xc.columns), class_names=["benigno", "maligno"], filled=True, fontsize=8)
plt.show()

## 3. Profundidad y sobreajuste (ejemplo 5.5)

In [ ]:
filas = []
for prof in [1, 2, 3, 5, None]:
    a = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(Xc_ent, yc_ent)
    filas.append({"profundidad": prof or "sin límite", "hojas": a.get_n_leaves(),
                  "entrenamiento": round(a.score(Xc_ent, yc_ent), 3), "prueba": round(a.score(Xc_pru, yc_pru), 3)})
pd.DataFrame(filas)

## 4. Un árbol frente a muchos (ejemplo 5.6)

El bosque de 300 árboles puede tardar unos segundos en el navegador.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
modelos = {
    "árbol": DecisionTreeClassifier(random_state=0),
    "bosque": RandomForestClassifier(n_estimators=300, random_state=0),
    "boosting": HistGradientBoostingClassifier(random_state=0),
}
for nombre, m in modelos.items():
    acc = cross_val_score(m, Xc, yc, cv=cv)
    print(f"{nombre:>9}: {acc.mean():.3f} ± {acc.std():.3f}")

### ¿Qué variables usa más el bosque?

In [ ]:
bosque = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xc_ent, yc_ent)
pd.Series(bosque.feature_importances_, index=Xc.columns).sort_values().tail(8).plot.barh(figsize=(6, 3.5))
plt.xlabel("importancia (reducción media de impureza)"); plt.show()

## 5. SVM: la escala importa (ejemplo 5.7)

In [ ]:
from sklearn.svm import SVC
con = cross_val_score(make_pipeline(StandardScaler(), SVC(kernel="rbf")), Xc, yc, cv=cv).mean()
sin = cross_val_score(SVC(kernel="rbf"), Xc, yc, cv=cv).mean()
print(f"con estandarización {con:.3f} · sin estandarizar {sin:.3f}")
Xc[["mean smoothness", "mean area"]].describe().loc[["min", "max"]]

**Ejercicio.** Prueba `SVC(kernel="rbf", C=c)` con `c` en 0,01, 0,1, 1, 10 y 100, siempre con estandarización. ¿Con qué valores empeora y por qué?

> Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
>
> Cortes, C. y Vapnik, V. (1995). Support-Vector Networks. *Machine Learning*, 20(3), 273–297. https://doi.org/10.1007/BF00994018